# Merchant Fraud-Risk Modelling

This notebook builds one retrospective risk-estimation row per merchant, uses the 61 directly labelled merchants as training targets, compares five models, and scores all 4,422 merchants. The output supports merchant-ranking review; it is not proof of fraud.

In [ ]:
import sys
import tempfile
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import Image, display

def find_member4_dir(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        direct = candidate if candidate.name == 'member4_fraud' else candidate / 'member4_fraud'
        if (direct / 'code').is_dir():
            return direct
    raise FileNotFoundError('Run this notebook from the repository or member4_fraud/code.')

MODULE_DIR = find_member4_dir()
CODE_DIR = MODULE_DIR / 'code'
REPO_ROOT = MODULE_DIR.parent
OUTPUT_DIR = Path(tempfile.gettempdir()) / 'member4_fraud_work' / 'merchant_model'
DELIVERY_DIR = MODULE_DIR / 'result'
RAW_TABLES = REPO_ROOT.parent / 'data' / 'part1' / 'tables'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print(sys.executable)

## 1. Modelling definition

The supplied merchant table has 114 dated scores for only 61 merchants. To match the final ranking grain and prevent the same merchant appearing in multiple splits, the dated scores are averaged to one target per merchant. `merchant_abn` and the direct fraud score are never input features.

Transaction features are calculated up to 28 February 2022. Consumer-day predictions are merged through `user_id` and aggregated into merchant exposure measures. Static merchant metadata is joined through `merchant_abn`.

## 2. Build the merchant feature tables

The scoring table contains every merchant. The training table is the 61-row labelled subset. Direct merchant fraud probability remains the target and is not used to create predictors.

In [ ]:
from build_merchant_model_table import build_merchant_model_table

merchant_table_summary = build_merchant_model_table(
    repo_root=REPO_ROOT, raw_tables_root=RAW_TABLES, output_dir=OUTPUT_DIR
)
display(merchant_table_summary)

In [ ]:
TRAINING_PATH = OUTPUT_DIR / 'merchant_fraud_training_table.parquet'
SCORING_PATH = OUTPUT_DIR / 'merchant_scoring_features_2022-02-28.parquet'
training_preview = duckdb.sql(f"""
    SELECT merchant_abn, fraud_probability, target_observations,
           log_total_revenue, repeat_consumer_share, sales_growth_log,
           amount_weighted_consumer_risk, high_risk_revenue_share
    FROM read_parquet('{TRAINING_PATH.as_posix()}') LIMIT 8
""").df()
display(training_preview)

## 3. New merged feature groups

**Merchant activity:** revenue, order value, consumer count, repeat share, recency, 30/90-day activity, growth, stability and high-value share.

**Consumer-risk exposure:** calculated for the later risk-combination stage, but excluded from KNN predictors to avoid counting the same consumer signal twice.

**Static merchant metadata:** take rate is retained. High-cardinality names, ABNs, categories and external demographics are excluded from this first model because 61 labels cannot support many additional parameters.

## 4. Validation and overfitting controls

- One row per merchant prevents repeated dates for the same merchant leaking across splits.
- 48 merchants form the model-selection pool; 13 stratified merchants are held out until the end.
- Hyperparameters are compared with repeated five-fold CV (5 folds × 10 repeats) on the 48 training merchants.
- Imputation and scaling are fitted inside each fold.
- Selection uses mean CV MAE, with CV RMSE as tie-breaker.
- Trees are shallow/regularized and KNN uses limited neighbour choices.
- The independent test set is evaluated only after every model form is fixed.

In [ ]:
from train_merchant_models import MODEL_FEATURES, run_merchant_model_experiment

merchant_model_comparison = run_merchant_model_experiment(
    repo_root=REPO_ROOT, output_dir=OUTPUT_DIR
)
display(merchant_model_comparison)

In [ ]:
split_summary = pd.read_csv(OUTPUT_DIR / 'merchant_data_split_summary.csv')
diagnostics = pd.read_csv(OUTPUT_DIR / 'merchant_feature_diagnostics.csv')
high_correlations = pd.read_csv(OUTPUT_DIR / 'merchant_high_correlation_pairs.csv')
vif = pd.read_csv(OUTPUT_DIR / 'merchant_vif.csv')
display(split_summary, diagnostics.sort_values('target_spearman', key=abs, ascending=False),
        high_correlations.head(20), vif.head(20))

## 5. Model 1 — Mean baseline

In [ ]:
tuning = pd.read_csv(OUTPUT_DIR / 'merchant_model_tuning_results.csv')
forms = pd.read_csv(OUTPUT_DIR / 'merchant_best_model_forms.csv')
display(tuning.query("model == 'Mean Baseline'"), forms.query("model == 'Mean Baseline'"))

## 6. Model 2 — K-nearest neighbours

The train-to-test error gap is reviewed because distance weighting can memorize a 48-merchant training set.

In [ ]:
display(tuning.query("model == 'KNN'").sort_values(['cv_mae_mean', 'cv_rmse_mean']),
        forms.query("model == 'KNN'"))

## 7. Model 3 — Linear regression

Coefficients are interpretable associations only; high VIF values warn where correlated merchant features make individual coefficients unstable.

In [ ]:
linear_coefficients = pd.read_csv(OUTPUT_DIR / 'merchant_linear_regression_coefficients.csv')
display(tuning.query("model == 'Linear Regression'"),
        forms.query("model == 'Linear Regression'"), linear_coefficients.head(15))

## 8. Model 4 — Random forest

Depth and leaf size are constrained to reduce small-sample overfitting.

In [ ]:
importance = pd.read_csv(OUTPUT_DIR / 'merchant_tree_feature_importance.csv')
display(tuning.query("model == 'Random Forest'").sort_values(['cv_mae_mean', 'cv_rmse_mean']),
        forms.query("model == 'Random Forest'"), importance.query("model == 'Random Forest'").head(12))

## 9. Model 5 — XGBoost

Only shallow, strongly regularized candidates are considered because the labelled sample is small.

In [ ]:
display(tuning.query("model == 'XGBoost'").sort_values(['cv_mae_mean', 'cv_rmse_mean']),
        forms.query("model == 'XGBoost'"), importance.query("model == 'XGBoost'").head(12))

## 10. Final model and outputs

KNN is fixed as the merchant scoring method. Repeated cross-validation chooses the neighbour count and weighting within KNN; the other model families are benchmarks only. The holdout result is reported as an independent check and is not used to switch models. Because only 61 merchants are labelled, uncertainty and prediction sensitivity must remain visible in the final ranking.

In [ ]:
comparison = pd.read_csv(OUTPUT_DIR / 'merchant_model_comparison.csv')
display(comparison, forms)
display(Image(filename=str(OUTPUT_DIR / 'figures' / 'merchant_model_comparison.png')))
selected_name = comparison.loc[comparison.selected_by_cv, 'model'].iloc[0]
if selected_name in {'Random Forest', 'XGBoost'}:
    display(Image(filename=str(OUTPUT_DIR / 'figures' / 'merchant_selected_model_feature_importance.png')))

In [ ]:
prediction_path = OUTPUT_DIR / 'merchant_fraud_predictions_all.parquet'
prediction_audit = duckdb.sql(f"""
    SELECT count(*) AS merchants,
           sum(merchant_fraud_label_available::INTEGER) AS directly_labelled_merchants,
           min(predicted_merchant_fraud_probability) AS min_prediction,
           avg(predicted_merchant_fraud_probability) AS mean_prediction,
           max(predicted_merchant_fraud_probability) AS max_prediction,
           count(DISTINCT predicted_merchant_fraud_probability) AS distinct_predictions
    FROM read_parquet('{prediction_path.as_posix()}')
""").df()
display(prediction_audit)
delivery_files = [
    'merchant_best_model_forms.csv',
    'merchant_data_split_summary.csv',
    'merchant_fraud_predictions_all.csv',
    'merchant_model_comparison.csv',
    'merchant_model_tuning_results.csv',
    'merchant_tree_feature_importance.csv',
]
DELIVERY_DIR.mkdir(parents=True, exist_ok=True)
for filename in delivery_files:
    source = OUTPUT_DIR / filename
    target = DELIVERY_DIR / filename
    target.write_bytes(source.read_bytes())
print('Delivered merchant CSV files to:', DELIVERY_DIR)

## 11. Responsible interpretation

The selected score is a model estimate learned from only 61 directly labelled merchants. It must be combined with revenue, growth, repeat behaviour and coverage in the final ranking rather than used as an automatic exclusion rule. Consumer and merchant fraud scores are supplied probabilities, not confirmed events.